# Lab info_01 — Entropy of English & Huffman compression

**Goals**
1. Estimate the entropy of English text from character frequencies (unigram) and from pairs (conditional bigram entropy).
2. Implement Huffman coding, encode/decode the text, and check $H \le \bar L < H + 1$.
3. Compare with a real compressor (`zlib`), which exploits context that a per-character code cannot.

Do the paper homework for **Shannon entropy** first — this lab assumes you can compute $H$ by hand.

**Reading**: Shannon (1948) §§1–7 and his estimates of English entropy;
MacKay, *Information Theory, Inference, and Learning Algorithms*, ch. 2 & 5 (https://www.inference.org.uk/itila/book.html);
MIT 6.050J units 1–5 (https://ocw.mit.edu/courses/6-050j-information-and-entropy-spring-2008/).

## Theory recap

* **Entropy** of a distribution $p$ (bits): $\displaystyle H(X) = -\sum_x p(x)\log_2 p(x)$, with $0\log 0 = 0$.
  It is at most $\log_2 |\mathcal X|$ (uniform). For a 27-symbol alphabet (a–z + space) that is $\log_2 27 \approx 4.75$ bits.
* **Conditional entropy** from pairs: $H(X_{i+1}\mid X_i) = H(X_i, X_{i+1}) - H(X_i)$. Conditioning never increases entropy,
  so context-aware coders can beat the unigram bound.
* **Huffman coding**: repeatedly merge the two least-frequent nodes; prepend `0`/`1` to the codewords of the merged subtrees.
  The result is an optimal prefix code: $H \le \bar L = \sum_x p(x)\ell(x) < H + 1$.
* **Kraft**: any prefix code satisfies $\sum_x 2^{-\ell(x)} \le 1$.

In [ ]:
import sys, math, re, zlib
from pathlib import Path
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

# make `lc_client` importable whether Jupyter was started in labs/ or in the repo root
LAB_DIR = Path.cwd() if (Path.cwd() / "lc_client.py").exists() else Path.cwd() / "labs"
sys.path.insert(0, str(LAB_DIR))
DATA = LAB_DIR / "data"

def normalize(text: str) -> str:
    """Lowercase, keep a-z and single spaces (27-symbol alphabet). Provided."""
    text = re.sub(r"[^a-z]+", " ", text.lower())
    return re.sub(r" +", " ", text).strip()

RAW = (DATA / "english_sample.txt").read_text(encoding="utf-8")
TEXT = normalize(RAW)
print(len(TEXT), "characters;", len(set(TEXT)), "distinct symbols")
print(TEXT[:200])

## 1. Entropy

Implement `entropy(p)` for a probability vector (list or array, may contain zeros), returning **bits**.
Then `unigram_entropy(text)` (character frequencies) and `conditional_bigram_entropy(text)` using the
consecutive pairs `zip(text, text[1:])`: $H(X_{i+1}\mid X_i) = H(\text{pairs}) - H(\text{first characters of the pairs})$.

In [ ]:
def entropy(p) -> float:
    """Shannon entropy in bits of a probability vector (zeros allowed)."""
    # TODO: ignore zero entries, return -sum p log2 p as a Python float
    raise NotImplementedError


def unigram_entropy(text: str) -> float:
    # TODO: empirical character distribution -> entropy
    raise NotImplementedError


def conditional_bigram_entropy(text: str) -> float:
    """H(X_{i+1} | X_i) estimated from consecutive character pairs."""
    # TODO: H(pairs) - H(text[:-1])   (both distributions over the len(text)-1 pairs)
    raise NotImplementedError

In [ ]:
H1 = unigram_entropy(TEXT)
H2 = conditional_bigram_entropy(TEXT)
print(f"max (uniform over {len(set(TEXT))}): {math.log2(len(set(TEXT))):.3f} bits/char")
print(f"unigram entropy      : {H1:.3f} bits/char")
print(f"H(next | previous)   : {H2:.3f} bits/char")

freqs = Counter(TEXT)
syms = sorted(freqs, key=freqs.get, reverse=True)
plt.figure(figsize=(9, 3))
plt.bar([repr(s)[1:-1] or "␣" for s in syms], [freqs[s] / len(TEXT) for s in syms])
plt.ylabel("p(char)"); plt.title("Character frequencies"); plt.show()

## 2. Huffman coding

Implement `huffman_code(freqs)` returning a dict `symbol -> bit string` (use `heapq`; break ties with a counter so the heap
never compares dicts). Handle the 1-symbol edge case (give it the code `"0"`). Then `encode`, `decode` and
`avg_code_length(code, freqs)` $= \sum_x p(x)\,\ell(x)$.

In [ ]:
import heapq, itertools

def huffman_code(freqs: dict) -> dict:
    """Binary Huffman code: symbol -> bit string."""
    # TODO: heap of (weight, tiebreak, {symbol: partial code}); merge the two lightest,
    #       prefixing "0" to one side's codes and "1" to the other's
    raise NotImplementedError


def encode(text: str, code: dict) -> str:
    # TODO
    raise NotImplementedError


def decode(bits: str, code: dict) -> str:
    # TODO: walk the bits, emitting a symbol whenever the buffer equals a codeword (prefix-free!)
    raise NotImplementedError


def avg_code_length(code: dict, freqs: dict) -> float:
    # TODO: sum_x p(x) * len(code[x])
    raise NotImplementedError

In [ ]:
CODE = huffman_code(freqs)
bits = encode(TEXT, CODE)
assert decode(bits, CODE) == TEXT
Lbar = avg_code_length(CODE, freqs)
kraft = sum(2.0 ** -len(c) for c in CODE.values())
print(f"Huffman: {Lbar:.3f} bits/char   (entropy {H1:.3f}; Kraft sum {kraft:.3f})")
for s in syms[:8]:
    print(repr(s), CODE[s])

## 3. Compare with zlib

`zlib` (LZ77 + Huffman) sees repeated *strings*, so it can go below the unigram entropy.
Fill the bar chart and explain in a markdown cell: why can a context model beat $H_1$, and why is even $H(X_{i+1}\mid X_i)$
far above Shannon's ~1 bit/char estimate for English?

In [ ]:
def zlib_bits_per_char(text: str) -> float:
    return len(zlib.compress(text.encode("ascii"), 9)) * 8 / len(text)

rates = {"uniform log2|A|": math.log2(len(set(TEXT))), "unigram H": H1, "Huffman": Lbar,
         "H(next|prev)": H2, "zlib -9": zlib_bits_per_char(TEXT)}
plt.figure(figsize=(7, 3))
plt.barh(list(rates), list(rates.values()))
plt.xlabel("bits per character"); plt.title("How many bits does English need?"); plt.show()
rates

*Your observations here.*

In [ ]:
from lc_client import grade
grade("info_01_entropy_compression", globals())